# Panacée — Pipeline complet Phase 1 → 2 → 3

Notebook pour serveur Linux distant (GPU) connecté à VSCode via Jupyter.

**Pipeline :**
- Phase 1 : Pré-entraînement MGM sur ZINC 250k (self-supervised)
- Phase 2 : Fine-tuning toxicité Tox21 (scaffold split)
- Phase 3 : Multi-propriétés + IA Raisonnement (Virtual Node, Bliss, etc.)

**Run All** pour tout lancer.

In [ ]:
# ═══ CONFIG ═══════════════════════════════════════════════════
REPO_URL = "https://github.com/jumoras0000/savh_gnn.git"
WORK_DIR = "/tmp/panacee"  # dossier de travail sur le serveur

# Epochs (réduire pour test rapide)
PHASE1_EPOCHS = 30
PHASE2_EPOCHS = 80
PHASE3_EPOCHS = 300

BATCH_SIZE = 128  # ajuster selon VRAM GPU (256 si >= 16GB, 64 si 8GB)
PHASE1_MAX_MOLECULES = 250000  # None = tout ZINC 250k

print("Config OK")

In [ ]:
# ═══ DIAGNOSTIC SERVEUR ══════════════════════════════════════
import os, subprocess, sys, shutil

print(f"Python  : {sys.version}")
print(f"OS      : {os.uname().sysname} {os.uname().release}")
print(f"CWD     : {os.getcwd()}")
print(f"User    : {os.environ.get('USER', 'unknown')}")

# GPU check
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
    print(f"GPU     : {gpu_info}")
except Exception:
    print("GPU     : ❌ nvidia-smi non trouvé — entraînement CPU (lent)")

# Disk
total, used, free = shutil.disk_usage("/tmp")
print(f"Disk /tmp: {free // (1024**3)} GB libre")

In [ ]:
# ═══ DÉPENDANCES ═════════════════════════════════════════════
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# PyTorch devrait déjà être installé sur le serveur GPU
try:
    import torch
    print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
except ImportError:
    print("❌ PyTorch non installé — installez-le d'abord (pip install torch)")
    raise

# torch-geometric + deps
try:
    import torch_geometric
    print(f"PyG {torch_geometric.__version__} ✓")
except ImportError:
    print("Installation torch-geometric...")
    pip('torch-geometric')

# RDKit + DeepChem
try:
    from rdkit import Chem
    print(f"RDKit ✓")
except ImportError:
    pip('rdkit')

try:
    import deepchem
    print(f"DeepChem {deepchem.__version__} ✓")
except ImportError:
    pip('deepchem')

# scikit-learn, tqdm
pip('scikit-learn', 'tqdm')
print("Dépendances ✓")

In [ ]:
# ═══ CLONER LE REPO ══════════════════════════════════════════
import subprocess, sys
from pathlib import Path

CLONE_DIR = Path(WORK_DIR)
PROJET = CLONE_DIR / "Projet_Panacee"

if not CLONE_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)], check=True)
    print(f"✓ Cloné → {CLONE_DIR}")
else:
    subprocess.run(['git', '-C', str(CLONE_DIR), 'pull', '--ff-only'], check=False)
    print(f"✓ Mis à jour → {CLONE_DIR}")

# Ajouter au path Python
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))

os.chdir(str(PROJET))
print(f"CWD: {os.getcwd()}")
print(f"Fichiers: {list(Path('.').glob('src/models/*.py'))}")

In [ ]:
# ═══ PHASE 1 — Pré-entraînement MGM (ZINC 250k) ═════════════
# Self-supervised : masque des atomes, le GNN apprend à les prédire.
# Produit : checkpoints/phase1/sovereign_encoder_v1.pth

import sys, os
os.environ['PYTHONIOENCODING'] = 'utf-8'

sys.argv = [
    'run_phase1.py',
    '--download',
    '--epochs', str(PHASE1_EPOCHS),
    '--batch_size', str(BATCH_SIZE),
]
if PHASE1_MAX_MOLECULES:
    sys.argv.extend(['--max_molecules', str(PHASE1_MAX_MOLECULES)])

print(f"Phase 1 : {PHASE1_EPOCHS} epochs, batch {BATCH_SIZE}")
print("="*60)

try:
    # Reload au cas où déjà importé
    if 'run_phase1' in sys.modules:
        del sys.modules['run_phase1']
    from run_phase1 import main as phase1_main
    phase1_main()
except SystemExit as e:
    if e.code and e.code != 0:
        print(f"⚠ Phase 1 exit: {e.code}")
    else:
        print("Phase 1 terminée.")

# Vérifier le checkpoint
from pathlib import Path
p1_ckpt = Path('checkpoints/phase1/sovereign_encoder_v1.pth')
if p1_ckpt.exists():
    print(f"\n✅ Phase 1 OK : {p1_ckpt} ({p1_ckpt.stat().st_size/1e6:.1f} MB)")
else:
    print(f"\n❌ Checkpoint Phase 1 introuvable!")

In [ ]:
# ═══ PHASE 2 — Fine-tuning Toxicité (Tox21) ═════════════════
# Transfer learning : encodeur Phase 1 → classification 12 tâches Tox21.
# Scaffold split pour évaluation honnête.
# Produit : checkpoints/phase2/best_toxicity_model.pth

import sys

sys.argv = [
    'run_phase2.py',
    '--download',
    '--epochs', str(PHASE2_EPOCHS),
    '--batch_size', str(BATCH_SIZE),
]

# Utiliser le checkpoint Phase 1 si disponible
p1_ckpt = Path('checkpoints/phase1/sovereign_encoder_v1.pth')
if p1_ckpt.exists():
    sys.argv.extend(['--pretrained_model', str(p1_ckpt)])
    print(f"Transfer depuis Phase 1 : {p1_ckpt}")
else:
    print("⚠ Pas de Phase 1 → Phase 2 from scratch (moins performant)")

print(f"Phase 2 : {PHASE2_EPOCHS} epochs, batch {BATCH_SIZE}")
print("="*60)

try:
    if 'run_phase2' in sys.modules:
        del sys.modules['run_phase2']
    from run_phase2 import main as phase2_main
    phase2_main()
except SystemExit as e:
    if e.code and e.code != 0:
        print(f"⚠ Phase 2 exit: {e.code}")
    else:
        print("Phase 2 terminée.")

p2_ckpt = Path('checkpoints/phase2/best_toxicity_model.pth')
if p2_ckpt.exists():
    print(f"\n✅ Phase 2 OK : {p2_ckpt} ({p2_ckpt.stat().st_size/1e6:.1f} MB)")
else:
    print(f"\n❌ Checkpoint Phase 2 introuvable!")

In [ ]:
# ═══ PHASE 3 — Multi-propriétés + IA Raisonnement ═══════════
# Transfer Phase 2 → prédiction multi-tâches :
#   Toxicité (12), Efficacité HIV, Solubilité, LogP, BBB, ClinTox
# + Module Raisonneur IA (Bliss independence, combinaisons)
# 
# Nouveautés v3 :
#   - Virtual Node (+1-3% OGB)
#   - TaskBalancedSampler (diversité tâches/batch)
#   - Uncertainty weighting (Kendall 2018)
#   - LLRD factor 0.8
#   - Bliss independence signal Raisonneur
# Produit : checkpoints/phase3/panacee_phase3_complete.pth

import sys

sys.argv = [
    'run_phase3.py',
    '--download',
    '--epochs', str(PHASE3_EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--skip_checks',
]

p2_ckpt = Path('checkpoints/phase2/best_toxicity_model.pth')
if p2_ckpt.exists():
    sys.argv.extend(['--pretrained_model', str(p2_ckpt)])
    print(f"Transfer depuis Phase 2 : {p2_ckpt}")
else:
    print("❌ Phase 2 manquante — relancez la cellule Phase 2")
    raise SystemExit("Phase 2 requise")

print(f"Phase 3 : {PHASE3_EPOCHS} epochs, batch {BATCH_SIZE}")
print("="*60)

try:
    if 'run_phase3' in sys.modules:
        del sys.modules['run_phase3']
    # Purger les modules src.* pour forcer le reload avec le code à jour
    for mod_name in list(sys.modules):
        if mod_name.startswith('src.'):
            del sys.modules[mod_name]
    from run_phase3 import main as phase3_main
    phase3_main()
except SystemExit as e:
    if e.code and e.code != 0:
        print(f"⚠ Phase 3 exit: {e.code}")
    else:
        print("Phase 3 terminée.")

p3_ckpt = Path('checkpoints/phase3/panacee_phase3_complete.pth')
if p3_ckpt.exists():
    print(f"\n✅ Phase 3 OK : {p3_ckpt} ({p3_ckpt.stat().st_size/1e6:.1f} MB)")
else:
    print(f"\n❌ Checkpoint Phase 3 introuvable!")

In [ ]:
# ═══ VÉRIFICATION FINALE ═════════════════════════════════════
from pathlib import Path
import torch

print("="*60)
print("RÉSUMÉ PIPELINE")
print("="*60)

checkpoints = {
    'Phase 1': 'checkpoints/phase1/sovereign_encoder_v1.pth',
    'Phase 2': 'checkpoints/phase2/best_toxicity_model.pth',
    'Phase 3': 'checkpoints/phase3/panacee_phase3_complete.pth',
}

for name, path in checkpoints.items():
    p = Path(path)
    if p.exists():
        size = p.stat().st_size / 1e6
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        keys = list(ckpt.keys()) if isinstance(ckpt, dict) else ['tensor']
        print(f"  ✅ {name}: {size:.1f} MB — clés: {keys[:5]}")
    else:
        print(f"  ❌ {name}: MANQUANT")

# Quick inference test
p3 = Path('checkpoints/phase3/panacee_phase3_complete.pth')
if p3.exists():
    print("\n--- Test rapide Phase 3 ---")
    from src.analysis.combinatorial_engine import PanaceeAnalyzer
    analyzer = PanaceeAnalyzer(checkpoint_path=str(p3))
    # Aspirine
    result = analyzer.predict_properties('CC(=O)Oc1ccccc1C(=O)O')
    if result:
        print(f"  Aspirine: safety={result['safety_score']}%, "
              f"efficacy={result['efficacy']['probabilite_activite']}%, "
              f"LogS={result['solubility']['log_s']}")
    # Test MC Dropout
    result_unc = analyzer.predict_properties('CC(=O)Oc1ccccc1C(=O)O', mc_samples=10)
    if result_unc and 'uncertainty' in result_unc:
        print(f"  MC Dropout: confidence={result_unc['uncertainty']['confidence_level']}")

print("\n" + "="*60)
print("PIPELINE COMPLET ✓")
print("="*60)

In [ ]:
# ═══ COPIER LES CHECKPOINTS (optionnel) ═════════════════════
# Copie les checkpoints dans un dossier accessible pour téléchargement
import shutil
from pathlib import Path

OUTPUT = Path('/tmp/panacee_output')
OUTPUT.mkdir(exist_ok=True)

for name, path in checkpoints.items():
    p = Path(path)
    if p.exists():
        dest = OUTPUT / p.name
        shutil.copy2(p, dest)
        print(f"  {name} → {dest}")

print(f"\nCheckpoints dans : {OUTPUT}")
print("Téléchargez-les via l'explorateur VSCode ou scp.")